# 1-0: How does AI actually "do" things? Reading the tool-calling note

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/1-0_tool_calling.ipynb)

> 📍 **Matching slides**: Session 1, "Stop 1: How AI gets things done", slides **5–15**; the usage-fields reading is appendix slide **45** (class time 0:10–0:35)

## What this notebook does

This is the first notebook of Session 1. It lays the foundation: **how does AI actually "do" things?**

A **model** (an LLM, a large language model, the thing behind ChatGPT and Gemini) first works by generating text: **it produces a reply from its input**.
A single model call only sees what you give it this time. Actions like browsing the web or running code need tools that your program provides. So why can ChatGPT run Python for you?
The answer is **tool calling**: the model only sends back a **note** saying "please run `run_code` with this code",
and the thing that actually runs it is your Python.

This notebook lets you see that note with your own eyes, then watch your Python follow the note and paste the result back.
Repeat that in a loop and you have an **agent**.

| Part | What you will see | Matching slides |
| --- | --- | --- |
| Part 1 | How relevant history changes the model's reply | 6–7 |
| Part 2 | What the model returns: text and tool requests; usage details are optional reading | 8; 45 (usage reading) |
| Part 3 | ⭐ The "note" the model sends back: `tool_calls` | 11–12 |
| Part 4 | Your Python follows the note and pastes the result back; put it in a loop and it is an agent | 13–14 |
| Part 5 | How a workflow differs from an agent (leads into 1-A and 1-B) | 15 |

## How to use it (you don't change anything in this notebook)

1. First run **"0. Environment setup"** below (click the ▶ to the left of the cell). The first run installs packages, and how long that takes depends on your connection and environment.
2. Every part follows the same order: **🤔 read the prediction questions and write your answers on paper → ▶ run → 👀 compare with "What did you see?"**.
3. **In class (required)**: the main output of each part. The usage details in Part 2 are optional reading (appendix slide 45); the instructor may demo them, and you do not need to memorize the fields.
4. Collapsed cells are code. Click "Show code" if you want to look; it's fine if you can't follow it, because in this session you only need to read the output.
5. After a runtime reset, rerun "0. Environment setup" first. After that, every cell builds the messages and tools it needs, so you can rerun just the cell you want. For Part 4, run the first cell and then the loop cell.

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** The foundation before the helper can "do" things. The model generates a reply; when it wants a tool, it only sends back a "note" naming the tool, and your Python is what actually does the work. The helpers in 1-A and 1-B are built on this.

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""The Homework Helper's toolbox: run_code, check_indent, read_spec, check_naming."""
''',
    'tools/run_code.py': r'''"""run_code: runs a student's homework under teaching restrictions.

A student's homework could contain anything (import os to delete files, an infinite loop), so this tool has two execution limits plus one readability feature:
1. A check before running: only a few standard modules may be imported (math, random, re, ...). os / sys / subprocess and the like are always blocked.
   eval, exec, open, input and attributes that start with an underscore (often used to get around restrictions) are not allowed either
2. The code runs in a separate child process, and it is stopped by force when the time runs out (so an infinite loop can be stopped too)
3. It returns only one line of result (which line, what error), which is easier for people and models to read. This is not a security limit

⚠️ This is a teaching fence. For a real product, use a real sandbox (a container, RestrictedPython, e2b, Modal).
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

BLOCKED_MODULES = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
ALLOWED_MODULES = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
BLOCKED_FUNCTIONS = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
BLOCKED_ATTRIBUTES = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
TIME_LIMIT = 3
MAX_OUTPUT_CHARS = 800

# The code in the child process that really runs the homework: it uses a "shrunken builtins", and import lets only the allowlist through
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("This tool does not allow import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "homework.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "homework.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _precheck(code: str) -> str | None:
    """A static check before running. If there is a problem, return a one-line explanation. Otherwise return None."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ Line {e.lineno} SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in BLOCKED_MODULES or root not in ALLOWED_MODULES:
                    return f"⛔ Blocked: line {node.lineno} import {name} (this tool does not allow {root})"
        elif isinstance(node, ast.Name) and node.id in BLOCKED_FUNCTIONS:
            return f"⛔ Blocked: line {node.lineno} uses {node.id} (this tool does not allow it)"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in BLOCKED_ATTRIBUTES or node.attr in BLOCKED_MODULES
        ):
            return f"⛔ Blocked: line {node.lineno} accesses .{node.attr} (this tool does not allow it)"
    return None


def run_homework(code: str, time_limit_seconds: float = TIME_LIMIT) -> str:
    """The function that does the real work: check → run in a child process → return one line of result."""
    if not code.strip():
        return "❌ The homework is empty, there is nothing to run"
    blocked = _precheck(code)
    if blocked:
        return blocked
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(ALLOWED_MODULES)), json.dumps(sorted(BLOCKED_FUNCTIONS))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=time_limit_seconds, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ Ran longer than {time_limit_seconds:g} seconds and was stopped by force (there may be an infinite loop)"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ The run failed (the program did not end normally; it may have used too much memory)"
    stdout = result.get("stdout", "")
    if len(stdout) > MAX_OUTPUT_CHARS:
        stdout = stdout[:MAX_OUTPUT_CHARS] + "\n... (output too long, the rest is omitted)"
    if result["ok"]:
        return f"✅ Ran successfully, output:\n{stdout}" if stdout else "✅ Ran successfully (nothing was printed)"
    where = f"line {result['line']} " if result.get("line") else ""
    output = f"\n(printed before the error: {stdout.strip()})" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{output}"


@tool
def run_code(code: str) -> str:
    """Run a piece of Python homework under teaching restrictions and return the output or the error message.

    When to use it: when you want to know what the homework actually prints when it runs, or whether it errors.
    Parameter code: the complete Python code (a string).
    Returns: one line of result, for example "✅ Ran successfully, output: 5" or "❌ line 3 NameError: ...".
    Limits: it can only import a few standard modules such as math, random and re; it cannot use input() or open(); it is stopped after 3 seconds.
    """
    return run_homework(code)
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

---
## Part 1: An LLM is a "continuation machine", and it has no memory

> 📍 **Matching slides**: slide 6 "An LLM is a "continuation machine"", slide 7 "LLMs have no memory: your program supplies the conversation"

Let's start with the basics: what does a model actually do?

Each time, the model sees only "the text you give it this time" and guesses the most sensible continuation, like a machine that finishes your sentence.
It does not remember the previous conversation, **unless your program passes the relevant history or a summary back in**.
The cell below asks "What is my name?" twice. The first time it passes the earlier conversation along; the second time it sends only that one sentence.

### 🤔 Prediction questions (write on paper before running)
1. The first time (with "My name is Alice" passed along), can it tell you your name?
2. The second time (only "What is my name?"), how will it answer?

In [ ]:
#@title ▶ Run: with history vs. without history
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain_core.messages import AIMessage, HumanMessage

# The LLM has no memory of its own, so the program passes the relevant history in
history = [
    HumanMessage(content="My name is Alice"),
    AIMessage(content="Hi Alice!"),
    HumanMessage(content="What is my name?"),
]
with_history = model.invoke(history).text
print("[Relevant history passed in]")
print(with_history)

# Counter-example: don't pass history, send only the last sentence
without_history = model.invoke("What is my name?").text
print("\n[Only the last sentence passed in]")
print(without_history)

record("1-0 Part 1: with history vs. without history", {"with_history": with_history, "without_history": without_history})

### 👀 What did you see?

- With history, the model has a basis for answering "Alice"; with only the last sentence, it has no way to know the name. Compare the actual outputs; the wording does not have to match.
- Same model, same question. The only difference is **whether you pasted the earlier lines back in**. It doesn't remember you; your program supplied the relevant record this time.
- Going further: how a conversation is merged in LangGraph State is the optional after-class part of 1-A; saving a conversation across runs is 2-D in Session 2.

---
## Part 2: Two kinds of content a model returns

> 📍 **Matching slides**: slide 8 "Tokens and the context window", slide 45 "Optional appendix: usage fields in the returned object"

The last part showed that the model rereads everything you give it every time. So how much can it read at once?

- **Token**: the unit in which a model processes text. How text is split depends on the model, and usage and billing are often counted in tokens.
- **Context window**: the maximum number of tokens a model can look at at once. Think of it as a **desk**: the conversation, instructions and data all have to be laid out on it, and if you put too much there, it won't fit.

A model returns a message object. Besides the content and tool requests, it may also include token usage.

### 🤔 Prediction questions
1. Will the model reply with text this time, or ask to use a tool? (Hint: we did not provide any tools.)
2. Remember two fields first: `content` is text, and `tool_calls` is a tool request. Which one do you predict will have something in it?

In [ ]:
#@title ▶ Run: look at the text and tool requests first (usage details are optional)
show_usage_details = False  #@param {type:"boolean"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
resp = model.invoke("Answer in three words: what is Python?")

print("content:", repr(resp.content))
print("tool_calls:", resp.tool_calls)
if show_usage_details:
    print("type:", type(resp).__name__)
    print("usage_metadata:", resp.usage_metadata)
    print("response_metadata keys:", list(resp.response_metadata.keys()))
    if "token_count" in resp.response_metadata:
        print("\n(The token count from this adapter is an estimate. It is not the usage reported by the provider.)")

record("1-0 Part 2: returned object", {"content": resp.content, "tool_calls": resp.tool_calls,
                          "usage_metadata": resp.usage_metadata})

### 👀 What did you see?

- This time, just look at `content` and `tool_calls`. You don't need to memorize the object type or the other fields.
- `.content` / `.text` is what it said; `.tool_calls` is an empty list, meaning it did **not** ask to use a tool this time (in the next part we provide a tool and then see whether it asks).
- **Optional reading**: if you tick the usage details, `usage_metadata` shows input and output token usage. Depending on the model source, it may report actual usage or a teaching estimate, so don't compare the two directly.
  Some models count reasoning tokens in their usage, so you can't work it out from the length of the visible answer alone.
- You don't need to remember the numbers. The optional MCP usage reading in 1-B compares the estimated length of tool descriptions.

---
## Part 3 ⭐: The model only "says" which tool it wants

> 📍 **Matching slides**: slide 11 "Tool calling: the model only "says" which tool it wants", slide 12 "Predict first: will the model ask to use a tool?"

The first two parts had no tools, so we only looked at text replies. But the helper needs to "do" things, like actually run a student's program. What then?
The answer is **tool calling**, which works like a three-panel comic:

1. Your Python tells the model: "You have a tool called `run_code` that can run code." ← this is `bind_tools`
2. The model returns a structured tool request (shown as JSON): "I want to use `run_code` with this code." ← this is `tool_calls`
3. Your Python sees the note, **actually runs it**, and pastes the result back to the model ← the next part

In this notebook we call the model's tool request the "note"; the "tool request" on the slides is the same thing.

- **Tool**: an ordinary Python function with one `@tool` line on top. Once you provide it to the model through `bind_tools`, the model receives its name, description and parameters.
- **JSON**: a data format written with curly braces `{}` that programs can read easily.

The `run_code` in this cell is fake and runs nothing; we only watch whether the model wants to call it.

### 🤔 Prediction questions
1. Will the model answer directly or ask to use a tool? If it answers directly, what will `content` (the text answer) say?
2. If it asks to use a tool, which tool do you predict it picks, and which code does it hand over? Read the field names after the output appears.
3. Did the model actually run `print('hi'`?

In [ ]:
#@title ▶ Run: look at the "note" the model sends back
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain_core.tools import tool

@tool
def run_code(code: str) -> str:
    """Run a Python homework and return its output or error message."""
    return "(Not actually run here; we only check whether the model wants to call it)"

model_with_tools = model.bind_tools([run_code])     # Comic panel 1: tell the model you have a tool
resp = model_with_tools.invoke("Does this homework run? print('hi'")

print("content:", repr(resp.content))
print("tool_calls:", resp.tool_calls)              # Comic panel 2: the JSON note

if "raw_text" in resp.response_metadata:
    print("\n📝 The raw text the model actually returned:")
    print(resp.response_metadata["raw_text"])
if not resp.tool_calls:
    print("\nThis time the model answered directly, with no tool request. Record this result; you can still compare with the note shown on the slides, and you don't need to keep rerunning.")

record("1-0 Part 3: the tool_calls note", {"content": resp.content, "tool_calls": resp.tool_calls})

### 👀 What did you see?

- **If there is a tool request**: first read `name` (which tool it picked) and `args` (what it handed to the tool). `content` may be an empty string.
- **If there is no tool request**: the model chose to answer directly. That is also a valid observation, not a failed run. Compare it with the tool-request example on slide 13.
- `tool_calls` is a list. Reading the tool name and arguments is enough for now; `id` is used to match results and `type` marks the kind of message, so treat both as optional reading.
- The model did **not** run `print('hi'`; it only "suggested" that you run it. Whether to run it, and how, is up to your program.
- With the built-in Colab model you will also see the "raw text": that is the response before the adapter parses it, and it may contain a tool request or a direct answer.
  On the Colab path, this course uses a teaching adapter (`review-bot/llm.py`) that writes the tool descriptions into the prompt and asks the model to reply in JSON,
  then turns that into an identical `tool_calls`. If you choose `gemini`, it uses native tool calling, and the note looks the same.
  This is exactly the point: **the model proposes a tool request, and your program does the work.**

---
## Part 4: Your Python follows the note, then pastes the result back

> 📍 **Matching slides**: slide 13 "Read the tool request, then call the model again", slide 14 "Agent = model + tools + loop"

The last part only observed whether the model proposes a tool request; it did not run the tool. In this part, if we get a request, we actually run it and send back the result.

This time we switch to a `run_code` that **really runs code**. It has guardrails: `import os` is not allowed, and it is stopped after 3 seconds (1-B explains this in detail).
The flow has three steps:

① Ask the model → ② Follow the note and run it, wrap the result in a `ToolMessage` (the kind of message that means "the tool's result") and paste it into the conversation → ③ Ask the model again

### 🤔 Prediction questions
1. In step ③, can the model see the result of `print(1+1)`? What will it say?
2. If you repeat ①②③ over and over, when should it stop?

In [ ]:
#@title ▶ Run: ① ask the model → ② actually run it → ③ paste the result back and ask again
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain_core.messages import HumanMessage, ToolMessage
from tools.run_code import run_code     # The version that really runs code (with guardrails)

model_with_tools = model.bind_tools([run_code])
messages = [HumanMessage("Does this homework run? Please actually run it and tell me the result.\n```python\nprint(1+1)\n```")]

# ① Ask the model
resp = model_with_tools.invoke(messages)
messages.append(resp)
print("① The model's note:", resp.tool_calls or "(no note, it answered directly)")

# ② Your Python follows the note and really runs it, then wraps the result in a ToolMessage and pastes it back
for call in resp.tool_calls:
    result = run_code.invoke(call["args"])
    print(f"② Your Python ran {call['name']}, result: {result}")
    messages.append(ToolMessage(content=result, tool_call_id=call["id"], name=call["name"]))

# ③ Ask the model again: this time it can see the result
if resp.tool_calls:
    final = model_with_tools.invoke(messages)
    messages.append(final)
    print("③ After seeing the result, the model said:", final.text or final.tool_calls)
else:
    print("This time the model did not call a tool, so step ② did not run. Record \"answered directly\" and compare with the slide example.")

record("1-0 Part 4: follow the note, run it, paste it back", messages)

### 👀 What did you see?

- When there is a tool request, in ② it is **your Python** that runs the code, not the model. The result is wrapped in a `ToolMessage` and matched to the note by `tool_call_id`.
- After the tool runs, in ③ the model can read the result, because we pasted it into the conversation (remember Part 1? Relevant results have to be added to the conversation by the program).
- Write ①②③ as a loop that keeps turning until the model says "no more tools", and that is an **agent**. Below is a loop with an upper limit that demonstrates this interaction.

### 🤔 Prediction questions
The loop below runs at most 5 rounds. How many rounds do you guess it will run before it stops? Why set "at most 5 rounds"?

In [ ]:
#@title ▶ Run: write the three steps as a loop, and that is an agent (model + tools + loop)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain_core.messages import HumanMessage, ToolMessage
from tools.run_code import run_code

model_with_tools = model.bind_tools([run_code])
messages = [HumanMessage("Check whether this homework runs, and if there is an error, tell me what is wrong.\n```python\nx = 3\nprint(x * y)\n```")]

for loop_no in range(1, 6):                         # At most 5 rounds, so it can't run forever
    resp = model_with_tools.invoke(messages)        # Ask the model
    messages.append(resp)
    if not resp.tool_calls:                         # It says no more tools -> done
        print(f"Round {loop_no}: the model said no more tools. Final answer:\n{resp.text}")
        break
    for call in resp.tool_calls:                    # It wants a tool -> run it and paste the result back
        result = run_code.invoke(call["args"])
        print(f"Round {loop_no}: the model called {call['name']} -> {result}")
        messages.append(ToolMessage(content=result, tool_call_id=call["id"], name=call["name"]))

else:
    print("Reached the maximum of 5 rounds and stopped; this does not mean the homework is finished.")

record("1-0 Part 4: agent loop", messages)

### 👀 What did you see?

- Round 1 might call `run_code` and round 2 might give the feedback; or it might answer directly, or use a tool again. Record the actual number of rounds and why it stopped.
- This loop shows the agent core model used in this course: **Agent = model + tools + loop**.
- "At most 5 rounds" is a limit your program sets on the AI: an agent can keep going in circles, and every round costs money. The `recursion_limit` you will see in 1-B (force a stop after this many steps) is also a stop limit, but it counts graph steps, not the loop rounds here.

---
## Part 5: Workflow vs. agent: who decides the next step?

> 📍 **Matching slides**: slide 15 "Workflow vs. agent: who decides the next step"

In the loop from the last part, **the model** decides on its own whether to use a tool next, which makes it an agent.
The other approach is to have your program arrange the steps, branches and stopping conditions. That is a **workflow**. Each has its strengths:

|  | Workflow | Agent |
| --- | --- | --- |
| Who decides the next step | The flow diagram you drew in code | The AI decides |
| What it's like | The order-fulfilment line in a bubble-tea shop | An intern holding a toolbox |
| Strength | Paths are easier to predict and debug | Can choose tools dynamically for the task |
| Weakness | You must design branches and exception handling up front | Less stable, may take detours |
| Where today | **1-A_workflow** | **1-B_agent** |

Next, 1-A builds a workflow helper whose route is decided by the program, and 1-B builds an agent helper that picks its own tools.

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()